## Resus Notebook Workflow Structure

This notebook systematically tests features extraction and retrieval functionality:

---
**Cells/Workflow Order:**

| Cell # | Purpose | Expected Results |
|--------|---------|------------------|
| 1-2 | Environment setup (imports, paths) | Python path configured correctly |
| 3 | Cleanup previous test outputs | No stale data remains |
| 4 | Start Elasticsearch container | Container running on port 9200 |
| 5 | Create credentials file | `test_elastic_credentials.py` generated |
| 6-7 | Populate dummy patient data | 5 patients in Elasticsearch cluster |
| 8 | Index refresh verification | All indices have documents |
| 9-10 | Initialize database and logging | Database at `outputs/temp_core_resus_db.sqlite` |
| 11-12 | Create pat2vec config with resus mode | Config object created successfully |
| 13-14 | Run pat2vec pipeline | Pipeline processes patients without errors |
| 15-16 | Extract all features from database | Features DataFrame populated |
| 17 | Output features dataframe preview | 3 rows of extracted features displayed |

| 18 | core_resus mode data retrieval test | `data` contains patient features, non-empty |

| 19-20 | Database and project cleanup | All temporary files deleted |
['| 21 | Final verification | All assertions pass |', '', '---', '**Test Failure Conditions:**', '- Any cell raises unhandled exception', '- Elasticsearch container fails to start', '- Empty patient list after population', '- Empty DataFrame returned from feature extraction', '']

In [ ]:
import os
import random
import shutil
import sys
from datetime import datetime

import numpy as np

random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

In [ ]:
current_dir = os.getcwd()
grandparent_dir = os.path.dirname(os.path.dirname(current_dir))

sys.path.insert(0, os.path.join(grandparent_dir, "pat2vec"))
sys.path.append(grandparent_dir)
pat2vec_dir = os.path.abspath(os.path.join(grandparent_dir, "pat2vec"))
sys.path.insert(0, pat2vec_dir)

print(f"Pat2vec path: {pat2vec_dir}")

In [ ]:
for dir_to_remove in ["bmi_test_project"]:
    try:
        shutil.rmtree(dir_to_remove, ignore_errors=True)
    except Exception as e:
        msg = (
            f"Failed to clean up '{dir_to_remove}' directory: {e}. "
            "Critical error - cannot start with stale data."
        )
        raise RuntimeError(
            msg,
        ) from e

print("Previous outputs cleaned.")

In [ ]:
from nb_temp_setup import get_notebook_port_offsetfrom pat2vec.util.docker_elastic import ElasticContainerport_offset = get_notebook_port_offset()es_container = ElasticContainer(port_offset=port_offset)es_container.stop()print("Starting Elasticsearch container (this may take a few seconds)...")if not es_container.start():    msg = "Failed to start Elasticsearch container. Check if Docker is running."    raise RuntimeError(        msg,    )host, username, password = es_container.get_credentials()creds_filename = "test_elastic_credentials.py"creds_content = f"""username = "{username}"password = "{password}"api_key = Nonehosts = ["{host}"]"""with open(creds_filename, "w") as f:    f.write(creds_content)print(f"Created '{creds_filename}' pointing to test cluster at {host}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

grandparent_dir = "/workspaces/pat2vec"
schema_path = os.path.join(grandparent_dir, "test_files", "elastic_schemas.json")

config_populate = config_class(
    proj_name="bmi_test_project",
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
)

In [ ]:
print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print()
print("Population complete.")
print(f"Generated {len(patient_ids)} dummy patients.")
print(f"Patient IDs: {patient_ids}")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = ["epr_documents", "basic_observations", "observations", "order", "pims_apps"]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
print("Indices refreshed.")

print()
print("Index Status:")
for index in indices:
    try:
        if cs.elastic.indices.exists(index=index):
            count = cs.elastic.count(index=index)["count"]
            print(f"  - {index:<20}: {count} documents")
        else:
            msg = f"Index not created: {index}"
            raise RuntimeError(msg)
    except Exception as e:
        msg = f"Error checking index {index}: {e}"
        raise RuntimeError(msg)

In [ ]:
PROJ_NAME = "bmi_test_project"
DB_FILENAME = "temp_bmi_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
except Exception as e:
    msg = (
        f"Failed to remove old database file '{DB_PATH}': {e}. "
        "Critical error - cannot start with stale data."
    )
    raise RuntimeError(
        msg,
    ) from e

db_connection_string = f"sqlite:///{DB_PATH}"
print(f"Database connection string set to: {db_connection_string}")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

config_obj = config_class(
    proj_name=PROJ_NAME,
    credentials_path=creds_filename,
    current_path_dir="",
    main_options={"bmi": True},
    batch_mode=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=None,
    patient_id_column_name="client_idcode",
    annot_filter_options={},
    shuffle_pat_list=False,
    storage_backend="database",
    check_patient_existence=False,
    db_connection_string=db_connection_string,
    treatment_doc_filename="test_files/treatment_docs.csv",
    all_patient_list=patient_ids,
)

print("pat2vec configuration created with BMI-only sources and database backend.")

In [ ]:
from pat2vec.main_pat2vec import main

try:
    pat2vec_obj = main(
        cogstack=True,
        use_filter=False,
        json_filter_path=None,
        random_seed_val=random_seed_value,
        hostname=None,
        config_obj=config_obj,
    )
except FileNotFoundError as e:
    msg = f"Failed to initialize pipeline: config path invalid. Error details: {e}."
    raise RuntimeError(
        msg,
    ) from e
except ValueError as e:
    msg = f"Failed to initialize pipeline: invalid configuration. Error details: {e}."
    raise RuntimeError(
        msg,
    ) from e
except RuntimeError as e:
    msg = f"Failed to initialize pipeline: initialization failure. Error details: {e}."
    raise RuntimeError(
        msg,
    ) from e
except Exception as e:
    msg = f"Failed to initialize pipeline: unexpected error. Error details: {e}."
    raise RuntimeError(
        msg,
    ) from e

print("pat2vec object initialized.")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

In [ ]:
if not pat2vec_obj.all_patient_list:
    msg = (
        "No patients in patient list after initialization. "
        "This indicates a critical failure in data loading or filtering."
    )
    raise RuntimeError(
        msg,
    )

print(f"Processing patient: {pat2vec_obj.all_patient_list[0]}")

try:
    pat2vec_obj.pat_maker(0)
except Exception as e:
    msg = (
        f"Failed to process patient 0 with pat_maker: {e}. "
        "Critical error - pipeline failed to extract features."
    )
    raise RuntimeError(
        msg,
    ) from e

print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    msg = (
        "FATAL ERROR: get_all_features returned an empty DataFrame. "
        "This indicates a critical failure in the pat2vec pipeline. "
        "No features were extracted or saved to the database."
    )
    raise RuntimeError(
        msg,
    )

print(f"Successfully retrieved {all_features.shape[0]} rows from database.")

In [ ]:
all_features_alt = pat2vec_obj.get_all_features()

if all_features_alt.empty:
    msg = (
        "FATAL ERROR: pat2vec_obj.get_all_features() returned an empty DataFrame. "
        "This indicates a critical failure in feature storage."
    )
    raise RuntimeError(
        msg,
    )

print(f"pat2vec_obj.get_all_features(): {all_features_alt.shape[0]} rows retrieved.")

In [ ]:
from pat2vec.util.post_processing import extract_datetime_to_column

df_with_datetime = extract_datetime_to_column(all_features)

print()
print("=== OUTPUT FEATURES DATAFRAME ===")
print(f"Shape: {df_with_datetime.shape}")
print(f"Total features: {len(df_with_datetime.columns)}")

if not df_with_datetime.empty:
    print()
    print("First 3 rows:")
    print(df_with_datetime.head(3))
else:
    msg = "DataFrame is empty after datetime extraction. Critical error - no features to extract."
    raise RuntimeError(
        msg,
    )

In [ ]:
print("\n=== DEMONSTRATING DATA RETRIEVAL FOR RESUS MODE ===")

all_pat_list = pat2vec_obj.all_patient_list

import pandas as pd

from pat2vec.pat2vec_get_methods.get_method_core_resus import get_core_resus

# Initialize empty DataFrame if needed
pat_batch = pd.DataFrame()

data = get_core_resus(
    current_pat_client_id_code=all_pat_list[0],
    target_date_range=(datetime(2020, 1, 1), datetime(2023, 12, 31)),
    pat_batch=pat_batch,
    config_obj=config_obj,
)

if data is None or (isinstance(data, list) and len(data) == 0):
    msg = (
        "FATAL ERROR: get_core_resus returned empty result. "
        "This indicates a critical failure in resus feature extraction."
    )
    raise RuntimeError(
        msg,
    )

patient_count = len(data) if isinstance(data, list) and len(data) > 0 else 1

print(f"Retrieved Resus data for {patient_count} patient(s)")
if isinstance(data, list):
    print(f"\nResus columns: {list(data[0].columns)}")
    print("\nSample data:")
    print(data[0])
else:
    print(f"\nResus columns: {list(data.columns)}")
    print("\nSample data:")
    print(data)

In [ ]:
# === VECTOR VALIDATION ===
feature_cols = [c for c in all_features.columns if c.startswith("core_resus_status_")]

assert len(feature_cols) > 0, "No feature columns found. Available columns: " + str(
    list(all_features.columns),
)

non_null_counts = all_features[feature_cols].notna().sum()
totally_empty = non_null_counts[non_null_counts == 0]

assert len(totally_empty) == 0, (
    f"The following feature columns are entirely null:\n"
    f"{list(totally_empty.index)}\n"
    "Vectorisation is silently failing — check the get method return value."
)

print("Feature columns (" + str(len(feature_cols)) + "): " + str(feature_cols))
print("Non-null counts per feature column:")
for col in sorted(feature_cols):
    val = all_features[col].notna().sum()
    print("  " + str(col) + ": " + str(val) + " non-null values")

In [ ]:
# Verify features were extracted correctly

print("\n=== VERIFYING FEATURE EXTRACTION ===")

features = pat2vec_obj.get_all_features()

if features.empty:
    msg = (
        "FATAL ERROR: get_all_features() returned empty DataFrame. "
        "This indicates a critical failure in feature storage."
    )
    raise RuntimeError(
        msg,
    )

print(f"Successfully retrieved {features.shape[0]} feature rows from database.")
print(f"Total features: {len(features.columns)}")

print("\n=== TEST SUCCESSFUL ===")